# Week 10: ANOVA & Non-parametric Methods — Applied Statistics with Python (2026)

Last week we used t-tests to compare **two** groups. But what if we have **three or more** groups? Running multiple t-tests inflates the Type I error rate (we learned this in Week 8). **ANOVA** (Analysis of Variance) solves this by testing all groups simultaneously. We also introduce **non-parametric methods** — distribution-free alternatives for when data doesn't meet the assumptions of parametric tests.

| # | Topic | Key Concepts |
|---|-------|--------------|
| 1 | Why ANOVA? | Multiple comparisons problem, F-test logic |
| 2 | One-Way ANOVA | Comparing means across k groups |
| 3 | ANOVA Assumptions | Normality, equal variances, independence |
| 4 | Post-Hoc Tests | Tukey HSD, Bonferroni, pairwise comparisons |
| 5 | Two-Way ANOVA | Two factors + interaction |
| 6 | Effect Size for ANOVA | Eta-squared, omega-squared |
| 7 | Mann-Whitney U Test | Non-parametric alternative to 2-sample t |
| 8 | Wilcoxon Signed-Rank | Non-parametric alternative to paired t |
| 9 | Kruskal-Wallis Test | Non-parametric alternative to one-way ANOVA |
| 10 | Parametric vs Non-parametric | Decision guide |
| 11 | Practical Example | Complete group comparison pipeline |
| 12 | Summary | Key functions and concepts |
| 13 | Homework | Practice exercises |

### Import all necessary libraries for this week's analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.anova import anova_lm

# Consistent style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
np.random.seed(42)

print('All libraries loaded successfully!')

---
## 1. Why ANOVA?

### The Multiple Comparisons Problem

Suppose we want to compare exam scores across 4 teaching methods. Running all pairwise t-tests gives us C(4,2) = 6 tests. With α = 0.05 each:

P(at least one false positive) = 1 - (1 - 0.05)⁶ = **26.5%**

ANOVA solves this by performing a **single omnibus test** at the chosen α level.

### The Logic of ANOVA

Despite its name, ANOVA compares **means** by analyzing **variance**:

| Source | Measures | If H₀ is true |
|--------|---------|---------------|
| **Between-group variance** (MS_between) | How much group means differ from the grand mean | Small (random only) |
| **Within-group variance** (MS_within) | How much individuals vary within each group | Same as between |

$$F = \frac{MS_{between}}{MS_{within}}$$

- If groups have the same mean → F ≈ 1
- If groups have different means → F >> 1

Demonstrate why running multiple t-tests inflates the false positive rate, motivating the need for ANOVA.

In [ ]:
# Simulation: multiple t-tests vs ANOVA when ALL groups have the same mean
np.random.seed(42)
n_sim = 10_000
k_groups = 4  # Number of groups
n_per_group = 30

# Count false positives
fp_multiple_t = 0  # At least one pairwise t-test significant
fp_anova = 0       # ANOVA F-test significant

for _ in range(n_sim):
    # Generate data: ALL groups have the SAME mean (H₀ is true)
    groups = [np.random.normal(50, 10, n_per_group) for _ in range(k_groups)]
    
    # Multiple t-tests: all pairs
    any_sig = False
    for i in range(k_groups):
        for j in range(i+1, k_groups):
            _, p = stats.ttest_ind(groups[i], groups[j])
            if p < 0.05:
                any_sig = True
                break
        if any_sig:
            break
    if any_sig:
        fp_multiple_t += 1
    
    # ANOVA: single test
    f_stat, p_anova = stats.f_oneway(*groups)
    if p_anova < 0.05:
        fp_anova += 1

print(f'=== False Positive Rates (H₀ true for all {k_groups} groups) ===')
print(f'Multiple t-tests: {fp_multiple_t/n_sim:.1%} (should be 5%, but inflated!)')
print(f'One-way ANOVA:    {fp_anova/n_sim:.1%} (correctly controlled at ~5%)')
print(f'\nTheoretical rate for {k_groups} groups, C({k_groups},2)={k_groups*(k_groups-1)//2} tests:')
print(f'  1 - (1-0.05)^{k_groups*(k_groups-1)//2} = {1-(0.95**(k_groups*(k_groups-1)//2)):.1%}')

Visualize the F-distribution and how the between-group vs within-group variance ratio determines significance.

In [ ]:
# Visualize the F-distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: F-distribution with rejection region
ax = axes[0]
df1, df2 = 3, 116  # 4 groups, 120 total observations
x = np.linspace(0, 6, 300)
y = stats.f.pdf(x, df1, df2)
f_crit = stats.f.ppf(0.95, df1, df2)

ax.plot(x, y, 'k-', linewidth=2)
ax.fill_between(x, y, where=(x >= f_crit), color='red', alpha=0.3, label=f'Rejection region (α=0.05)')
ax.fill_between(x, y, where=(x < f_crit), color='steelblue', alpha=0.2, label='Fail to reject')
ax.axvline(f_crit, color='red', linestyle='--', linewidth=1.5)
ax.text(f_crit + 0.1, 0.02, f'F_crit = {f_crit:.2f}', fontsize=11, color='red')
ax.set_xlabel('F-statistic', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'F-Distribution (df1={df1}, df2={df2})', fontsize=13)
ax.legend(fontsize=10)

# Right: visual example — same vs different group means
ax = axes[1]
x_vals = np.linspace(20, 80, 200)

# H₀ true: all means = 50
for i, color in enumerate(['steelblue', 'coral', 'seagreen']):
    ax.plot(x_vals, stats.norm.pdf(x_vals, 50, 10), color=color, linewidth=1.5, alpha=0.5)
ax.text(50, 0.042, 'H₀: All means equal', ha='center', fontsize=11, fontweight='bold')

# H₁ true: different means
means = [40, 50, 60]
for m, color in zip(means, ['steelblue', 'coral', 'seagreen']):
    ax.plot(x_vals, stats.norm.pdf(x_vals, m, 10) - 0.06, color=color, linewidth=2)
ax.text(50, -0.02, 'H₁: Means differ', ha='center', fontsize=11, fontweight='bold')
ax.axhline(0, color='gray', linewidth=0.5)

ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('ANOVA: Do Group Means Differ?', fontsize=13)
ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## 2. One-Way ANOVA

**One-way ANOVA** tests whether the means of **k groups** are equal when there is **one grouping factor**.

- H₀: μ₁ = μ₂ = ... = μₖ (all group means are equal)
- H₁: At least one mean is different

### ANOVA Table

| Source | SS | df | MS | F |
|--------|----|----|----|-|
| Between | SS_B = Σnⱼ(x̄ⱼ - x̄)² | k - 1 | SS_B / (k-1) | MS_B / MS_W |
| Within | SS_W = ΣΣ(xᵢⱼ - x̄ⱼ)² | N - k | SS_W / (N-k) | |
| Total | SS_T = ΣΣ(xᵢⱼ - x̄)² | N - 1 | | |

### 2.1 Basic One-Way ANOVA

A pharmaceutical company tests 4 different pain medications. Does pain relief differ across drugs?

Generate pain relief data for 4 drug groups and perform a one-way ANOVA.

In [ ]:
# Pain relief scores (0-10) for 4 drugs
np.random.seed(42)
drug_a = np.random.normal(5.0, 1.5, 30)  # Placebo-like
drug_b = np.random.normal(6.5, 1.5, 30)  # Moderate
drug_c = np.random.normal(7.0, 1.5, 30)  # Good
drug_d = np.random.normal(5.5, 1.5, 30)  # Slight improvement

# Clip to valid range
for arr in [drug_a, drug_b, drug_c, drug_d]:
    arr[:] = arr.clip(0, 10)

# Create DataFrame
pain_data = pd.DataFrame({
    'relief': np.concatenate([drug_a, drug_b, drug_c, drug_d]),
    'drug': ['Drug A']*30 + ['Drug B']*30 + ['Drug C']*30 + ['Drug D']*30
})

# Method 1: scipy.stats.f_oneway()
f_stat, p_value = stats.f_oneway(drug_a, drug_b, drug_c, drug_d)

print('=== One-Way ANOVA ===')
print(f'Group means:')
print(pain_data.groupby('drug')['relief'].agg(['mean', 'std', 'count']).round(3))
print(f'\nF-statistic: {f_stat:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'\nDecision at α=0.05: {"Reject H₀" if p_value < 0.05 else "Fail to reject H₀"}')
if p_value < 0.05:
    print('→ At least one drug has a significantly different mean pain relief.')

### 2.2 ANOVA with statsmodels (Full ANOVA Table)

For a proper ANOVA table with SS, df, MS, and F, use `statsmodels`.

Fit the ANOVA model using statsmodels to get the complete ANOVA table with sum of squares decomposition.

In [ ]:
# Method 2: statsmodels — gives full ANOVA table
model_anova = smf.ols('relief ~ C(drug)', data=pain_data).fit()
anova_table = anova_lm(model_anova, typ=1)

print('=== ANOVA Table ===')
print(anova_table.round(4))

# Manual verification
grand_mean = pain_data['relief'].mean()
ss_between = sum(30 * (pain_data.groupby('drug')['relief'].mean() - grand_mean)**2)
ss_within = sum(pain_data.groupby('drug')['relief'].apply(lambda x: sum((x - x.mean())**2)))
ss_total = sum((pain_data['relief'] - grand_mean)**2)

print(f'\n=== Manual Verification ===')
print(f'SS_between = {ss_between:.2f}')
print(f'SS_within  = {ss_within:.2f}')
print(f'SS_total   = {ss_total:.2f} (= {ss_between:.2f} + {ss_within:.2f} = {ss_between+ss_within:.2f})')

Visualize the drug comparison with both box plots and individual data points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot with individual points (swarm)
ax = axes[0]
sns.boxplot(data=pain_data, x='drug', y='relief', ax=ax, palette='Set2', 
            width=0.5, fliersize=0)
sns.stripplot(data=pain_data, x='drug', y='relief', ax=ax, color='black', 
              alpha=0.3, size=4, jitter=True)
ax.set_xlabel('Drug', fontsize=12)
ax.set_ylabel('Pain Relief Score', fontsize=12)
ax.set_title(f'Pain Relief by Drug (F = {f_stat:.2f}, p = {p_value:.4f})', fontsize=13)

# Means with CI
ax = axes[1]
group_stats = pain_data.groupby('drug')['relief'].agg(['mean', 'std', 'count'])
group_stats['se'] = group_stats['std'] / np.sqrt(group_stats['count'])
group_stats['ci'] = group_stats['se'] * stats.t.ppf(0.975, group_stats['count'] - 1)

colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']
ax.barh(range(4), group_stats['mean'], xerr=group_stats['ci'], 
        color=colors, alpha=0.7, height=0.6, capsize=5)
ax.axvline(grand_mean, color='red', linestyle='--', linewidth=2, label=f'Grand mean = {grand_mean:.2f}')
ax.set_yticks(range(4))
ax.set_yticklabels(group_stats.index, fontsize=11)
ax.set_xlabel('Pain Relief Score', fontsize=12)
ax.set_title('Group Means with 95% CI', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 3. ANOVA Assumptions

One-way ANOVA assumes:

| Assumption | How to Check | What if Violated? |
|-----------|-------------|-------------------|
| **Independence** | Study design | Use repeated-measures ANOVA |
| **Normality** (within each group) | Shapiro-Wilk, Q-Q plot | Use Kruskal-Wallis |
| **Equal variances** (homoscedasticity) | Levene's test, Bartlett's test | Use Welch's ANOVA |

ANOVA is **robust** to mild violations of normality (especially with equal group sizes), but sensitive to unequal variances.

Check all three ANOVA assumptions for our pain relief data.

In [ ]:
print('=== ANOVA Assumption Checks ===\n')

# 1. Normality within each group (Shapiro-Wilk)
print('1. Normality (Shapiro-Wilk per group):')
for drug_name in ['Drug A', 'Drug B', 'Drug C', 'Drug D']:
    data = pain_data[pain_data['drug'] == drug_name]['relief']
    stat, p = stats.shapiro(data)
    print(f'   {drug_name}: W = {stat:.4f}, p = {p:.4f} → {"Normal" if p > 0.05 else "Non-normal"}')

# 2. Equal variances (Levene's test)
groups = [pain_data[pain_data['drug'] == d]['relief'] for d in pain_data['drug'].unique()]
lev_stat, lev_p = stats.levene(*groups)
print(f'\n2. Equal Variances (Levene\'s test):')
print(f'   F = {lev_stat:.4f}, p = {lev_p:.4f} → {"Equal" if lev_p > 0.05 else "Unequal"} variances')

# Bartlett's test (more powerful but assumes normality)
bart_stat, bart_p = stats.bartlett(*groups)
print(f'\n   Bartlett\'s test: χ² = {bart_stat:.4f}, p = {bart_p:.4f}')

# Group variances
print(f'\n   Group standard deviations:')
print(pain_data.groupby('drug')['relief'].std().round(3).to_string())

Visualize the normality check with Q-Q plots for each group.

In [ ]:
# Q-Q plots for each group
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, drug_name in zip(axes, ['Drug A', 'Drug B', 'Drug C', 'Drug D']):
    data = pain_data[pain_data['drug'] == drug_name]['relief']
    stats.probplot(data, dist='norm', plot=ax)
    ax.set_title(drug_name, fontsize=12)

plt.suptitle('Q-Q Plots: Checking Normality by Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.1 Welch's ANOVA

When equal variance assumption is violated, use **Welch's ANOVA** — it doesn't assume equal variances.

Demonstrate Welch's ANOVA as a robust alternative when group variances are unequal.

In [ ]:
# Create data with unequal variances
np.random.seed(42)
group_equal_var = [np.random.normal(50, 5, 30), np.random.normal(53, 5, 30), 
                   np.random.normal(48, 5, 30)]
group_unequal_var = [np.random.normal(50, 3, 30), np.random.normal(53, 8, 30), 
                     np.random.normal(48, 15, 30)]  # Very different variances!

# Standard ANOVA
f1, p1 = stats.f_oneway(*group_unequal_var)

# Welch's ANOVA (scipy doesn't have it directly, but pingouin does)
# Manual: use Alexander-Govern test or one-way Welch
# Using scipy: stats.alexandergovern for Welch-like ANOVA
result_welch = stats.alexandergovern(*group_unequal_var)

print('=== Unequal Variance Example ===')
print(f'Group std devs: {[g.std():.2f for g in group_unequal_var]}')
lev, lev_p = stats.levene(*group_unequal_var)
print(f'Levene\'s test: p = {lev_p:.4f} → Variances are {"un" if lev_p < 0.05 else ""}equal')

print(f'\nStandard ANOVA:  F = {f1:.4f}, p = {p1:.4f}')
print(f'Alexander-Govern (Welch-like): T = {result_welch.statistic:.4f}, p = {result_welch.pvalue:.4f}')
print(f'\n→ When variances are unequal, use Welch\'s ANOVA for reliable results.')

---
## 4. Post-Hoc Tests

ANOVA tells us *that* at least one group differs, but not *which* groups differ. **Post-hoc tests** perform pairwise comparisons while controlling the Type I error rate.

| Method | Controls | Use When |
|--------|---------|----------|
| **Tukey HSD** | Family-wise error rate | Equal group sizes, equal variances |
| **Bonferroni** | Family-wise error rate | Any situation (conservative) |
| **Games-Howell** | Per-comparison | Unequal variances/sizes |
| **Dunnett's** | Family-wise | Comparing all groups to a control |

**Only run post-hoc tests if ANOVA is significant** (p < α).

### 4.1 Tukey's HSD (Honestly Significant Difference)

The most common post-hoc test. It tests all pairwise differences while controlling the overall Type I error.

Perform Tukey's HSD test to identify which specific drug pairs have significantly different pain relief.

In [ ]:
# Tukey's HSD
tukey = pairwise_tukeyhsd(pain_data['relief'], pain_data['drug'], alpha=0.05)
print('=== Tukey HSD Post-Hoc Test ===')
print(tukey)
print(f'\nSignificant differences (p < 0.05):')
tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], 
                        columns=tukey._results_table.data[0])
sig = tukey_df[tukey_df['reject'] == True]
if len(sig) > 0:
    for _, row in sig.iterrows():
        print(f'  {row["group1"]} vs {row["group2"]}: diff = {row["meandiff"]:.3f}, p = {row["p-adj"]:.4f}')
else:
    print('  None')

Visualize the Tukey HSD results with a confidence interval plot.

In [ ]:
# Tukey HSD visualization
fig, ax = plt.subplots(figsize=(10, 6))
tukey.plot_simultaneous(ax=ax)
ax.set_xlabel('Pain Relief Score', fontsize=12)
ax.set_title('Tukey HSD: 95% Simultaneous Confidence Intervals\n'
             '(Groups with non-overlapping intervals are significantly different)', fontsize=13)
plt.tight_layout()
plt.show()

### 4.2 Bonferroni Correction (Pairwise T-Tests)

An alternative to Tukey: run all pairwise t-tests but apply Bonferroni correction to the p-values.

Perform pairwise t-tests with Bonferroni correction and display results in a matrix format.

In [ ]:
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# All pairwise t-tests
drug_names = ['Drug A', 'Drug B', 'Drug C', 'Drug D']
drug_data = {name: pain_data[pain_data['drug'] == name]['relief'].values for name in drug_names}

pairs = []
p_values_raw = []
for d1, d2 in combinations(drug_names, 2):
    _, p = stats.ttest_ind(drug_data[d1], drug_data[d2])
    pairs.append((d1, d2))
    p_values_raw.append(p)

# Bonferroni correction
reject_bonf, pvals_bonf, _, _ = multipletests(p_values_raw, method='bonferroni')

print('=== Pairwise T-Tests with Bonferroni Correction ===')
print(f'{"Comparison":>25}  {"Raw p":>10}  {"Adjusted p":>12}  {"Significant":>12}')
print('-' * 65)
for (d1, d2), p_raw, p_adj, rej in zip(pairs, p_values_raw, pvals_bonf, reject_bonf):
    print(f'{d1} vs {d2:>8}  {p_raw:>10.4f}  {p_adj:>12.4f}  {"Yes *" if rej else "No":>12}')

---
## 5. Two-Way ANOVA

**Two-way ANOVA** tests the effects of **two factors** simultaneously, plus their **interaction**.

Example: Does exam performance depend on teaching method AND study time?

| Effect | Tests Whether |
|--------|---------------|
| **Main effect A** | Factor A affects the outcome (averaging over B) |
| **Main effect B** | Factor B affects the outcome (averaging over A) |
| **Interaction A×B** | The effect of A depends on the level of B |

Generate data with two factors (teaching method and class size) and perform a two-way ANOVA.

In [ ]:
# Two-way ANOVA: Teaching Method × Class Size → Exam Score
np.random.seed(42)
n_per_cell = 25  # Balanced design

data_rows = []
for method in ['Lecture', 'Flipped', 'Online']:
    for size in ['Small', 'Large']:
        # Base score depends on method
        base = {'Lecture': 70, 'Flipped': 75, 'Online': 68}[method]
        # Small classes do slightly better
        size_effect = 3 if size == 'Small' else 0
        # Interaction: Flipped classroom benefits MORE from small class
        interaction = 5 if (method == 'Flipped' and size == 'Small') else 0
        
        scores = np.random.normal(base + size_effect + interaction, 8, n_per_cell)
        for s in scores:
            data_rows.append({'method': method, 'class_size': size, 'score': s})

two_way_data = pd.DataFrame(data_rows)

# Two-way ANOVA with interaction
model_2way = smf.ols('score ~ C(method) * C(class_size)', data=two_way_data).fit()
anova_2way = anova_lm(model_2way, typ=2)  # Type II for unbalanced designs

print('=== Two-Way ANOVA ===')
print(anova_2way.round(4))

print(f'\nInterpretation:')
for effect in ['C(method)', 'C(class_size)', 'C(method):C(class_size)']:
    p = anova_2way.loc[effect, 'PR(>F)']
    name = effect.replace('C(', '').replace(')', '').replace(':', ' × ')
    print(f'  {name}: p = {p:.4f} → {"Significant" if p < 0.05 else "Not significant"}')

Visualize the main effects and interaction with an interaction plot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Main effect: Method
ax = axes[0]
sns.boxplot(data=two_way_data, x='method', y='score', ax=ax, palette='Set2',
            order=['Lecture', 'Flipped', 'Online'])
ax.set_title('Main Effect: Teaching Method', fontsize=13)
ax.set_xlabel('Method', fontsize=12)
ax.set_ylabel('Exam Score', fontsize=12)

# 2. Main effect: Class Size
ax = axes[1]
sns.boxplot(data=two_way_data, x='class_size', y='score', ax=ax, palette='Set3')
ax.set_title('Main Effect: Class Size', fontsize=13)
ax.set_xlabel('Class Size', fontsize=12)
ax.set_ylabel('Exam Score', fontsize=12)

# 3. Interaction plot
ax = axes[2]
interaction = two_way_data.groupby(['method', 'class_size'])['score'].mean().unstack()
interaction = interaction[['Small', 'Large']]  # Reorder
interaction.plot(marker='o', linewidth=2.5, markersize=8, ax=ax)
ax.set_xlabel('Teaching Method', fontsize=12)
ax.set_ylabel('Mean Exam Score', fontsize=12)
ax.set_title('Interaction: Method × Class Size', fontsize=13)
ax.legend(title='Class Size', fontsize=10)

plt.tight_layout()
plt.show()

print('Cell means:')
print(two_way_data.groupby(['method', 'class_size'])['score'].mean().round(2).unstack())

---
## 6. Effect Size for ANOVA

Just like with t-tests, we need to report how *big* the effect is, not just whether it's significant.

| Measure | Formula | Small | Medium | Large |
|---------|---------|-------|--------|-------|
| **η² (eta-squared)** | SS_between / SS_total | 0.01 | 0.06 | 0.14 |
| **ω² (omega-squared)** | (SS_B - df_B·MS_W) / (SS_T + MS_W) | 0.01 | 0.06 | 0.14 |
| **partial η²** | SS_effect / (SS_effect + SS_error) | Same | Same | Same |

η² is simpler but **positively biased** (overestimates). ω² corrects for this.

Compute eta-squared and omega-squared for our one-way ANOVA on drug effectiveness.

In [ ]:
def anova_effect_size(anova_table):
    """Compute eta-squared and omega-squared from ANOVA table."""
    ss_between = anova_table.iloc[0]['sum_sq']
    ss_within = anova_table.iloc[1]['sum_sq']
    ss_total = ss_between + ss_within
    df_between = anova_table.iloc[0]['df']
    ms_within = anova_table.iloc[1]['mean_sq'] if 'mean_sq' in anova_table.columns else ss_within / anova_table.iloc[1]['df']
    
    eta_sq = ss_between / ss_total
    omega_sq = (ss_between - df_between * ms_within) / (ss_total + ms_within)
    
    return eta_sq, omega_sq

eta2, omega2 = anova_effect_size(anova_table)

print('=== Effect Size for One-Way ANOVA (Drug Study) ===')
print(f'η² (eta-squared):   {eta2:.4f}')
print(f'ω² (omega-squared): {omega2:.4f}')
print(f'\nInterpretation: {eta2:.1%} of the variance in pain relief is explained by drug type.')
print(f'Effect size: {"Small" if eta2 < 0.06 else "Medium" if eta2 < 0.14 else "Large"}')

# For two-way ANOVA: partial eta-squared for each effect
print(f'\n=== Partial η² for Two-Way ANOVA ===')
ss_error = anova_2way.loc['Residual', 'sum_sq']
for effect in ['C(method)', 'C(class_size)', 'C(method):C(class_size)']:
    ss_effect = anova_2way.loc[effect, 'sum_sq']
    partial_eta2 = ss_effect / (ss_effect + ss_error)
    name = effect.replace('C(', '').replace(')', '').replace(':', ' × ')
    print(f'  {name:>20}: partial η² = {partial_eta2:.4f}')

---
## 7. Mann-Whitney U Test

The **Mann-Whitney U test** (also called Wilcoxon rank-sum test) is the non-parametric alternative to the **independent two-sample t-test**.

| Feature | Two-sample t-test | Mann-Whitney U |
|---------|------------------|----------------|
| Assumes normality | Yes | No |
| Compares | Means | Distributions (rank-based) |
| Sensitive to | Mean differences | Any distributional shift |
| Power | Higher (if normal) | Lower (but robust) |

- H₀: The two distributions are identical
- H₁: One distribution is stochastically greater than the other

Compare the t-test and Mann-Whitney U test on both normal and skewed data to see when each is appropriate.

In [ ]:
np.random.seed(42)

# Scenario 1: Normal data → t-test and Mann-Whitney agree
normal_a = np.random.normal(50, 10, 30)
normal_b = np.random.normal(55, 10, 30)

t_stat, t_p = stats.ttest_ind(normal_a, normal_b)
u_stat, u_p = stats.mannwhitneyu(normal_a, normal_b, alternative='two-sided')

print('=== Scenario 1: Normal Data ===')
print(f'T-test:       t = {t_stat:.4f}, p = {t_p:.4f}')
print(f'Mann-Whitney: U = {u_stat:.1f}, p = {u_p:.4f}')

# Scenario 2: Skewed data → Mann-Whitney is more appropriate
skewed_a = np.random.exponential(5, 30)
skewed_b = np.random.exponential(8, 30)

t_stat2, t_p2 = stats.ttest_ind(skewed_a, skewed_b)
u_stat2, u_p2 = stats.mannwhitneyu(skewed_a, skewed_b, alternative='two-sided')

print(f'\n=== Scenario 2: Skewed Data (Exponential) ===')
print(f'T-test:       t = {t_stat2:.4f}, p = {t_p2:.4f}')
print(f'Mann-Whitney: U = {u_stat2:.1f}, p = {u_p2:.4f}')

# Scenario 3: Ordinal data → only Mann-Whitney is valid
ordinal_a = np.random.choice([1, 2, 3, 4, 5], size=40, p=[0.30, 0.30, 0.20, 0.15, 0.05])
ordinal_b = np.random.choice([1, 2, 3, 4, 5], size=40, p=[0.10, 0.15, 0.25, 0.30, 0.20])

u_stat3, u_p3 = stats.mannwhitneyu(ordinal_a, ordinal_b, alternative='two-sided')
print(f'\n=== Scenario 3: Ordinal Rating Data ===')
print(f'Median A: {np.median(ordinal_a):.1f}, Median B: {np.median(ordinal_b):.1f}')
print(f'Mann-Whitney: U = {u_stat3:.1f}, p = {u_p3:.4f}')

Visualize the three scenarios with both distributions overlaid.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Normal data
axes[0].hist(normal_a, bins=12, alpha=0.6, color='steelblue', label='Group A', density=True)
axes[0].hist(normal_b, bins=12, alpha=0.6, color='coral', label='Group B', density=True)
axes[0].set_title(f'Normal Data\nMW p = {u_p:.4f}', fontsize=12)
axes[0].legend(fontsize=10)

# Skewed data
axes[1].hist(skewed_a, bins=12, alpha=0.6, color='steelblue', label='Group A', density=True)
axes[1].hist(skewed_b, bins=12, alpha=0.6, color='coral', label='Group B', density=True)
axes[1].set_title(f'Skewed Data\nMW p = {u_p2:.4f}', fontsize=12)
axes[1].legend(fontsize=10)

# Ordinal data
width = 0.35
x_ord = np.arange(1, 6)
count_a = [np.sum(ordinal_a == i) for i in range(1, 6)]
count_b = [np.sum(ordinal_b == i) for i in range(1, 6)]
axes[2].bar(x_ord - width/2, count_a, width, alpha=0.7, color='steelblue', label='Group A')
axes[2].bar(x_ord + width/2, count_b, width, alpha=0.7, color='coral', label='Group B')
axes[2].set_title(f'Ordinal Data\nMW p = {u_p3:.4f}', fontsize=12)
axes[2].set_xlabel('Rating')
axes[2].legend(fontsize=10)

plt.suptitle('Mann-Whitney U Test: Works for Any Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Wilcoxon Signed-Rank Test

The **Wilcoxon signed-rank test** is the non-parametric alternative to the **paired t-test**. It tests whether the median of paired differences equals zero.

- Uses **ranks of absolute differences** rather than raw values
- Does not assume normality of differences
- Good for ordinal data or small, skewed samples

Compare the paired t-test and Wilcoxon signed-rank test on a before/after therapy scenario with skewed improvements.

In [ ]:
# Before/after therapy: anxiety scores (skewed improvement)
np.random.seed(42)
n = 20
before_therapy = np.random.normal(70, 12, n).clip(30, 100)
# Improvement is right-skewed (most improve a little, some improve a lot)
improvement = np.random.exponential(8, n)
after_therapy = (before_therapy - improvement).clip(10, 100)

differences = before_therapy - after_therapy

# Paired t-test
t_stat, t_p = stats.ttest_rel(before_therapy, after_therapy)

# Wilcoxon signed-rank test
w_stat, w_p = stats.wilcoxon(before_therapy, after_therapy, alternative='greater')

print('=== Before/After Therapy: Anxiety Scores ===')
print(f'Mean before: {before_therapy.mean():.1f}')
print(f'Mean after:  {after_therapy.mean():.1f}')
print(f'Median difference: {np.median(differences):.1f}')

print(f'\nPaired t-test:    t = {t_stat:.4f}, p = {t_p:.4f}')
print(f'Wilcoxon signed-rank: W = {w_stat:.1f}, p = {w_p:.4f}')

# Check normality of differences
shap_stat, shap_p = stats.shapiro(differences)
print(f'\nShapiro-Wilk on differences: p = {shap_p:.4f}')
print(f'Differences are {"" if shap_p > 0.05 else "NOT "}normally distributed.')
if shap_p < 0.05:
    print('→ Wilcoxon signed-rank is the more appropriate test here.')

Visualize the paired differences and their distribution to see the non-normality.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Before/After paired lines
ax = axes[0]
for i in range(n):
    color = 'green' if after_therapy[i] < before_therapy[i] else 'red'
    ax.plot([0, 1], [before_therapy[i], after_therapy[i]], color=color, alpha=0.5, linewidth=1.5)
    ax.plot([0, 1], [before_therapy[i], after_therapy[i]], 'o', color=color, markersize=4)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Before', 'After'], fontsize=12)
ax.set_ylabel('Anxiety Score', fontsize=12)
ax.set_title('Individual Changes', fontsize=13)

# Distribution of differences
ax = axes[1]
ax.hist(differences, bins=10, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='No change')
ax.axvline(np.median(differences), color='orange', linewidth=2, label=f'Median = {np.median(differences):.1f}')
ax.set_xlabel('Improvement (Before - After)', fontsize=11)
ax.set_title('Distribution of Differences\n(Right-skewed)', fontsize=13)
ax.legend(fontsize=10)

# Q-Q plot of differences
stats.probplot(differences, plot=axes[2])
axes[2].set_title(f'Q-Q Plot of Differences\n(Shapiro p = {shap_p:.4f})', fontsize=13)

plt.tight_layout()
plt.show()

---
## 9. Kruskal-Wallis Test

The **Kruskal-Wallis test** is the non-parametric alternative to **one-way ANOVA**. It compares distributions of **three or more independent groups** using ranks.

- H₀: All groups come from the same distribution
- H₁: At least one group differs

If significant, follow up with **Dunn's test** for pairwise comparisons (the non-parametric equivalent of Tukey HSD).

### 9.1 Kruskal-Wallis Example

Compare customer satisfaction ratings (ordinal 1-5) across three stores.

Perform a Kruskal-Wallis test on ordinal satisfaction ratings across three stores.

In [ ]:
# Customer satisfaction (1-5 Likert scale) across 3 stores
np.random.seed(42)
store_a = np.random.choice([1, 2, 3, 4, 5], size=50, p=[0.05, 0.15, 0.30, 0.35, 0.15])
store_b = np.random.choice([1, 2, 3, 4, 5], size=45, p=[0.10, 0.20, 0.35, 0.25, 0.10])
store_c = np.random.choice([1, 2, 3, 4, 5], size=55, p=[0.03, 0.07, 0.20, 0.40, 0.30])

# Kruskal-Wallis test
h_stat, kw_p = stats.kruskal(store_a, store_b, store_c)

# Also run ANOVA for comparison
f_stat, anova_p = stats.f_oneway(store_a, store_b, store_c)

print('=== Customer Satisfaction by Store ===')
print(f'Store A: median = {np.median(store_a):.0f}, mean = {store_a.mean():.2f} (n={len(store_a)})')
print(f'Store B: median = {np.median(store_b):.0f}, mean = {store_b.mean():.2f} (n={len(store_b)})')
print(f'Store C: median = {np.median(store_c):.0f}, mean = {store_c.mean():.2f} (n={len(store_c)})')

print(f'\nKruskal-Wallis: H = {h_stat:.4f}, p = {kw_p:.4f}')
print(f'One-way ANOVA:  F = {f_stat:.4f}, p = {anova_p:.4f}')
print(f'\n→ Kruskal-Wallis is more appropriate for ordinal data.')

### 9.2 Dunn's Post-Hoc Test

If Kruskal-Wallis is significant, use Dunn's test for pairwise comparisons.

Perform Dunn's test with Bonferroni correction to identify which store pairs differ.

In [ ]:
# Dunn's test: pairwise Mann-Whitney with Bonferroni correction
stores = {'Store A': store_a, 'Store B': store_b, 'Store C': store_c}
store_names = list(stores.keys())

print('=== Pairwise Mann-Whitney (Dunn-like) with Bonferroni ===')
pairs = []
p_vals = []
for i, (n1, d1) in enumerate(stores.items()):
    for j, (n2, d2) in enumerate(stores.items()):
        if i < j:
            _, p = stats.mannwhitneyu(d1, d2, alternative='two-sided')
            pairs.append((n1, n2))
            p_vals.append(p)

# Bonferroni correction
reject, p_adj, _, _ = multipletests(p_vals, method='bonferroni')

for (s1, s2), p_raw, p_corr, rej in zip(pairs, p_vals, p_adj, reject):
    print(f'  {s1} vs {s2}: p_raw = {p_raw:.4f}, p_adj = {p_corr:.4f} '
          f'→ {"Significant *" if rej else "Not significant"}')

Visualize the store satisfaction comparison with violin plots showing the distribution shapes.

In [ ]:
# Create DataFrame for plotting
store_data = pd.DataFrame({
    'satisfaction': np.concatenate([store_a, store_b, store_c]),
    'store': ['Store A']*len(store_a) + ['Store B']*len(store_b) + ['Store C']*len(store_c)
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
ax = axes[0]
sns.violinplot(data=store_data, x='store', y='satisfaction', ax=ax, palette='Set2', inner='box')
ax.set_title(f'Satisfaction Distribution by Store\nKruskal-Wallis p = {kw_p:.4f}', fontsize=13)
ax.set_xlabel('Store', fontsize=12)
ax.set_ylabel('Satisfaction Rating', fontsize=12)

# Stacked bar chart (percentage)
ax = axes[1]
rating_counts = store_data.groupby(['store', 'satisfaction']).size().unstack(fill_value=0)
rating_pct = rating_counts.div(rating_counts.sum(axis=1), axis=0) * 100
rating_pct.plot(kind='bar', stacked=True, ax=ax, colormap='RdYlGn', width=0.6)
ax.set_title('Rating Distribution (%)', fontsize=13)
ax.set_xlabel('Store', fontsize=12)
ax.set_ylabel('Percentage', fontsize=12)
ax.legend(title='Rating', fontsize=9, bbox_to_anchor=(1.02, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

---
## 10. Parametric vs Non-parametric: Decision Guide

### When to Use Which?

| Scenario | Parametric Test | Non-parametric Alternative |
|----------|----------------|---------------------------|
| 1 sample, compare to value | One-sample t | Wilcoxon signed-rank (1-sample) |
| 2 independent groups | Two-sample t | Mann-Whitney U |
| 2 paired groups | Paired t | Wilcoxon signed-rank |
| 3+ independent groups | One-way ANOVA | Kruskal-Wallis |
| 3+ paired groups | Repeated-measures ANOVA | Friedman test |

### Decision Flowchart Rules:
1. **Ordinal data?** → Non-parametric
2. **Small n (< 15) and non-normal?** → Non-parametric
3. **Heavily skewed with outliers?** → Non-parametric
4. **n ≥ 30 and moderately non-normal?** → Parametric is usually fine (CLT)
5. **Normal data?** → Parametric (more powerful)

Create a simulation comparing the power of parametric vs non-parametric tests under different distribution assumptions.

In [ ]:
# Power comparison: parametric vs non-parametric
np.random.seed(42)
n_sim = 5000
n = 20
effect = 0.5  # Cohen's d = 0.5

distributions = {
    'Normal': lambda n, shift: np.random.normal(shift, 1, n),
    'Exponential': lambda n, shift: np.random.exponential(1, n) + shift,
    'Heavy-tailed (t, df=3)': lambda n, shift: np.random.standard_t(3, n) + shift,
}

results = []
for dist_name, gen_func in distributions.items():
    t_power, mw_power = 0, 0
    for _ in range(n_sim):
        a = gen_func(n, 0)
        b = gen_func(n, effect)
        
        _, p_t = stats.ttest_ind(a, b)
        _, p_mw = stats.mannwhitneyu(a, b, alternative='two-sided')
        
        if p_t < 0.05: t_power += 1
        if p_mw < 0.05: mw_power += 1
    
    results.append({
        'Distribution': dist_name,
        'T-test Power': t_power / n_sim,
        'Mann-Whitney Power': mw_power / n_sim
    })

power_df = pd.DataFrame(results)
print('=== Power Comparison (n=20, d=0.5, 5000 simulations) ===')
print(power_df.to_string(index=False))
print('\nKey insights:')
print('• Normal data: t-test is slightly more powerful')
print('• Skewed/heavy-tailed: Mann-Whitney can be MORE powerful')
print('• Non-parametric tests are not always less powerful!')

Visualize the power comparison across distributions.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_pos = np.arange(len(power_df))
width = 0.35

ax.bar(x_pos - width/2, power_df['T-test Power'], width, color='steelblue', 
       alpha=0.8, label='T-test')
ax.bar(x_pos + width/2, power_df['Mann-Whitney Power'], width, color='coral', 
       alpha=0.8, label='Mann-Whitney')

ax.set_xticks(x_pos)
ax.set_xticklabels(power_df['Distribution'], fontsize=11)
ax.set_ylabel('Power (proportion of rejections)', fontsize=12)
ax.set_title('Statistical Power: Parametric vs Non-parametric', fontsize=14)
ax.legend(fontsize=12)
ax.set_ylim(0, 1)
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='Target power = 0.80')

# Annotate values
for i, row in power_df.iterrows():
    ax.text(i - width/2, row['T-test Power'] + 0.02, f'{row["T-test Power"]:.2f}', 
            ha='center', fontsize=10)
    ax.text(i + width/2, row['Mann-Whitney Power'] + 0.02, f'{row["Mann-Whitney Power"]:.2f}', 
            ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## 11. Practical Example: Comparing Marketing Campaign Effectiveness

A retail company ran 4 different marketing campaigns across stores in 3 regions. They want to know:
1. Do campaigns differ in effectiveness?
2. Does effectiveness vary by region?
3. Is there an interaction between campaign and region?

### Step 1: Generate and Explore the Data

Create a realistic marketing dataset with sales lift data across campaigns and regions.

In [ ]:
np.random.seed(42)

campaigns = ['Email', 'Social Media', 'TV Ad', 'Direct Mail']
regions = ['North', 'South', 'West']

data_rows = []
camp_effects = {'Email': 8, 'Social Media': 12, 'TV Ad': 15, 'Direct Mail': 6}
region_effects = {'North': 2, 'South': 0, 'West': -1}

for camp in campaigns:
    for region in regions:
        n_stores = np.random.randint(12, 18)
        base = camp_effects[camp] + region_effects[region]
        # Add interaction: TV Ad works better in North
        if camp == 'TV Ad' and region == 'North':
            base += 4
        sales_lift = np.random.normal(base, 4, n_stores)
        for s in sales_lift:
            data_rows.append({'campaign': camp, 'region': region, 'sales_lift': s})

marketing = pd.DataFrame(data_rows)

print('=== Marketing Campaign Data ===')
print(f'Total observations: {len(marketing)}')
print(f'\nMean sales lift by campaign:')
print(marketing.groupby('campaign')['sales_lift'].mean().round(2).sort_values(ascending=False))
print(f'\nMean sales lift by region:')
print(marketing.groupby('region')['sales_lift'].mean().round(2).sort_values(ascending=False))

### Step 2: Visual Exploration

Visualize the data with box plots and an interaction plot to guide our analysis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By campaign
sns.boxplot(data=marketing, x='campaign', y='sales_lift', ax=axes[0], 
            palette='Set2', order=['Direct Mail', 'Email', 'Social Media', 'TV Ad'])
axes[0].set_title('Sales Lift by Campaign', fontsize=13)
axes[0].set_ylabel('Sales Lift (%)', fontsize=12)
axes[0].tick_params(axis='x', rotation=15)

# By region
sns.boxplot(data=marketing, x='region', y='sales_lift', ax=axes[1], palette='Set3')
axes[1].set_title('Sales Lift by Region', fontsize=13)
axes[1].set_ylabel('Sales Lift (%)', fontsize=12)

# Interaction plot
interaction_means = marketing.groupby(['campaign', 'region'])['sales_lift'].mean().reset_index()
for camp in campaigns:
    camp_data = interaction_means[interaction_means['campaign'] == camp]
    axes[2].plot(camp_data['region'], camp_data['sales_lift'], 'o-', 
                linewidth=2, markersize=8, label=camp)
axes[2].set_title('Interaction: Campaign × Region', fontsize=13)
axes[2].set_ylabel('Mean Sales Lift (%)', fontsize=12)
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

### Step 3: Statistical Tests

Run one-way ANOVA, two-way ANOVA, and Kruskal-Wallis, then perform post-hoc comparisons.

In [ ]:
# 1. One-way ANOVA: campaign effect
groups_camp = [marketing[marketing['campaign'] == c]['sales_lift'] for c in campaigns]
f1, p1 = stats.f_oneway(*groups_camp)
print('=== One-Way ANOVA: Campaign Effect ===')
print(f'F = {f1:.4f}, p = {p1:.6f}')

# 2. One-way ANOVA: region effect
groups_reg = [marketing[marketing['region'] == r]['sales_lift'] for r in regions]
f2, p2 = stats.f_oneway(*groups_reg)
print(f'\n=== One-Way ANOVA: Region Effect ===')
print(f'F = {f2:.4f}, p = {p2:.4f}')

# 3. Two-way ANOVA with interaction
model_2w = smf.ols('sales_lift ~ C(campaign) * C(region)', data=marketing).fit()
anova_2w = anova_lm(model_2w, typ=2)
print(f'\n=== Two-Way ANOVA ===')
print(anova_2w.round(4))

# 4. Kruskal-Wallis (non-parametric check)
h_stat, kw_p = stats.kruskal(*groups_camp)
print(f'\n=== Kruskal-Wallis (Campaign) ===')
print(f'H = {h_stat:.4f}, p = {kw_p:.6f}')

### Step 4: Post-Hoc and Effect Sizes

Run Tukey HSD for campaign comparisons and compute effect sizes.

In [ ]:
# Tukey HSD for campaigns
tukey_camp = pairwise_tukeyhsd(marketing['sales_lift'], marketing['campaign'], alpha=0.05)
print('=== Tukey HSD: Campaign Pairwise Comparisons ===')
print(tukey_camp)

# Effect sizes
eta2_camp = anova_2w.loc['C(campaign)', 'sum_sq'] / (anova_2w['sum_sq'].sum())
eta2_reg = anova_2w.loc['C(region)', 'sum_sq'] / (anova_2w['sum_sq'].sum())
eta2_inter = anova_2w.loc['C(campaign):C(region)', 'sum_sq'] / (anova_2w['sum_sq'].sum())

print(f'\n=== Effect Sizes (η²) ===')
print(f'Campaign:    η² = {eta2_camp:.4f} ({"Small" if eta2_camp < 0.06 else "Medium" if eta2_camp < 0.14 else "Large"})')
print(f'Region:      η² = {eta2_reg:.4f} ({"Small" if eta2_reg < 0.06 else "Medium" if eta2_reg < 0.14 else "Large"})')
print(f'Interaction: η² = {eta2_inter:.4f} ({"Small" if eta2_inter < 0.06 else "Medium" if eta2_inter < 0.14 else "Large"})')

### Step 5: Summary Report

Compile a comprehensive summary with recommendations.

In [ ]:
print('=' * 60)
print('  MARKETING CAMPAIGN EFFECTIVENESS — ANALYSIS REPORT')
print('=' * 60)

print(f'\n1. CAMPAIGN EFFECT')
print(f'   One-way ANOVA: F = {f1:.2f}, p < 0.001 → Significant')
print(f'   η² = {eta2_camp:.3f} → Large effect')
print(f'   Ranking (mean sales lift):')
ranking = marketing.groupby('campaign')['sales_lift'].mean().sort_values(ascending=False)
for i, (camp, mean_val) in enumerate(ranking.items(), 1):
    print(f'     {i}. {camp}: {mean_val:.1f}%')

print(f'\n2. REGION EFFECT')
p_reg = anova_2w.loc['C(region)', 'PR(>F)']
print(f'   Two-way ANOVA: p = {p_reg:.4f} → {"Significant" if p_reg < 0.05 else "Not significant"}')

print(f'\n3. INTERACTION (Campaign × Region)')
p_inter = anova_2w.loc['C(campaign):C(region)', 'PR(>F)']
print(f'   Two-way ANOVA: p = {p_inter:.4f} → {"Significant" if p_inter < 0.05 else "Not significant"}')
print(f'   TV Ads are especially effective in the North region.')

print(f'\n4. RECOMMENDATIONS')
print(f'   • TV Ad is the most effective campaign overall')
print(f'   • TV Ad + North region is the best combination')
print(f'   • Consider investing more in TV for North region stores')
print(f'   • Direct Mail shows lowest effectiveness — consider discontinuing')
print('=' * 60)

---
## 12. Summary

| Section | Key Concepts | Python Functions |
|---------|-------------|------------------|
| 1. Why ANOVA | Multiple comparisons inflate Type I error | — |
| 2. One-Way ANOVA | Compare k group means, F-test | `stats.f_oneway()`, `anova_lm()` |
| 3. Assumptions | Normality, equal variance, independence | `stats.shapiro()`, `stats.levene()` |
| 4. Post-Hoc | Tukey HSD, Bonferroni pairwise | `pairwise_tukeyhsd()`, `multipletests()` |
| 5. Two-Way ANOVA | Two factors + interaction | `smf.ols('y ~ C(A) * C(B)')` |
| 6. Effect Size | η², ω², partial η² | Manual computation |
| 7. Mann-Whitney U | Non-parametric 2-sample | `stats.mannwhitneyu()` |
| 8. Wilcoxon | Non-parametric paired | `stats.wilcoxon()` |
| 9. Kruskal-Wallis | Non-parametric k-sample | `stats.kruskal()` |
| 10. Decision Guide | Parametric vs non-parametric | Depends on data |
| 11. Practical | Complete campaign analysis | All above combined |

### Key Takeaways

1. **Use ANOVA** (not multiple t-tests) when comparing 3+ groups
2. **Post-hoc tests** tell you *which* groups differ — only run after significant ANOVA
3. **Two-way ANOVA** reveals main effects and interactions between factors
4. **Non-parametric tests** (Mann-Whitney, Kruskal-Wallis) are safe alternatives for ordinal or non-normal data
5. **Check assumptions first** — Levene's test for variances, Shapiro-Wilk for normality
6. **Always report effect size** — η² tells you how *much* the groups differ, not just whether they do

---
## 13. Homework

### Task 1: One-Way ANOVA + Post-Hoc
A teacher uses three grading methods (A: curve, B: absolute, C: rubric) across different classes. Generate data:
```python
np.random.seed(42)
method_a = np.random.normal(78, 8, 35)  # Curve grading
method_b = np.random.normal(72, 10, 30) # Absolute grading  
method_c = np.random.normal(82, 7, 32)  # Rubric grading
```
1. Check ANOVA assumptions (normality per group, Levene's test)
2. Perform one-way ANOVA
3. If significant, run Tukey HSD and identify which methods differ
4. Compute η² and interpret the effect size
5. Visualize with box plots + individual points and Tukey CI plot

### Task 2: Two-Way ANOVA
An experiment studies the effect of **fertilizer type** (Organic, Chemical, None) and **water level** (Low, Medium, High) on plant growth (cm). Create a balanced design with 15 plants per cell:
1. Generate data where fertilizer and water both affect growth, with an interaction (Chemical fertilizer benefits more from high water)
2. Perform two-way ANOVA with interaction
3. Create an interaction plot
4. Compute partial η² for each effect
5. Run post-hoc comparisons for the significant main effects

### Task 3: Non-parametric Tests
Customer ratings (1-10) for two versions of a product:
```python
np.random.seed(42)
version_old = np.random.choice(range(1, 11), size=45, p=[0.05, 0.08, 0.12, 0.15, 0.20, 0.15, 0.10, 0.08, 0.05, 0.02])
version_new = np.random.choice(range(1, 11), size=50, p=[0.02, 0.03, 0.05, 0.10, 0.15, 0.20, 0.18, 0.12, 0.10, 0.05])
```
1. Test normality — can we use a t-test?
2. Perform both a t-test and Mann-Whitney U test — do they agree?
3. Compute medians and mean ranks for each group
4. Visualize with histograms and violin plots
5. Which test is more appropriate for this data and why?

### Task 4: Complete Analysis Pipeline
Create a dataset of employee productivity scores across 4 departments and 2 shifts (Day/Night), n=20 per cell.
1. Explore the data (descriptive stats, visualizations)
2. Check ANOVA assumptions
3. Run two-way ANOVA — are there main effects and/or interaction?
4. If assumptions are violated, also run Kruskal-Wallis on the main factors
5. Perform appropriate post-hoc tests
6. Write a 4-sentence summary of your findings

---
### Next Week Preview

**Week 11: Time Series Analysis** — Many real-world datasets have a **time dimension**: stock prices, temperature readings, monthly sales. We'll learn to decompose time series into trend, seasonality, and residual components, test for stationarity, and build forecasting models using moving averages, exponential smoothing, and ARIMA.

---
*Applied Statistics with Python (2026) | Week 10 | ANOVA & Non-parametric Methods*